<a href="https://colab.research.google.com/github/rafa1485/clases_ds_ia/blob/Benner_Davila/Modelos_de_busqueda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import heapq

# Grafo y costos definidos en la consigna
grafo = {
    'S': {'A': 2, 'B': 2},
    'A': {'C': 2, 'D': 5},
    'B': {'D': 2},
    'C': {'G': 3},
    'D': {'G': 6},
    'G': {}
}

def ucs(inicio, objetivo):
    frontera = []
    orden_insercion = 0 # Contador para garantizar el desempate FIFO

    # El nodo almacena: estado, padre y g (costo acumulado)
    raiz = {"estado": inicio, "padre": None, "g": 0}
    mejor_g = {inicio: 0}

    # frontera guarda tuplas: (prioridad g, orden, nodo)
    heapq.heappush(frontera, (0, orden_insercion, raiz))
    orden_insercion += 1

    expandidos = 0

    print("--- Traza de Expansión UCS ---")

    while frontera:
        # Extraemos el nodo con menor prioridad g (y desempate por inserción)
        g_actual, _, nodo = heapq.heappop(frontera)
        estado = nodo["estado"]

        # Descartar entradas obsoletas frente a un mejor g ya conocido
        if nodo["g"] > mejor_g.get(estado, float('inf')):
            continue

        print(f"Extraído: {estado} | Costo acumulado: {nodo['g']}")
        expandidos += 1

        # El objetivo se comprueba al extraer, no al generar
        if estado == objetivo:
            # Reconstrucción del camino desde los padres
            camino = []
            actual = nodo
            while actual:
                camino.append(actual["estado"])
                actual = actual["padre"]
            return camino[::-1], nodo["g"], expandidos

        # Relajar transiciones legales
        for vecino, costo_arista in grafo.get(estado, {}).items():
            nuevo_g = nodo["g"] + costo_arista

            # Solo insertamos si mejoramos el costo conocido (mejor_g)
            if nuevo_g < mejor_g.get(vecino, float('inf')):
                mejor_g[vecino] = nuevo_g
                nuevo_nodo = {"estado": vecino, "padre": nodo, "g": nuevo_g}
                heapq.heappush(frontera, (nuevo_g, orden_insercion, nuevo_nodo))
                orden_insercion += 1

    return None, 0, expandidos

camino, costo, expandidos = ucs('S', 'G')
print(f"\nResultado -> Camino: {camino} | Costo total: {costo} | Estados expandidos: {expandidos}")

--- Traza de Expansión UCS ---
Extraído: S | Costo acumulado: 0
Extraído: A | Costo acumulado: 2
Extraído: B | Costo acumulado: 2
Extraído: C | Costo acumulado: 4
Extraído: D | Costo acumulado: 4
Extraído: G | Costo acumulado: 7

Resultado -> Camino: ['S', 'A', 'C', 'G'] | Costo total: 7 | Estados expandidos: 6


In [ ]:
import heapq

grafo = {
    'S': {'A': 2, 'B': 2},
    'A': {'C': 2, 'D': 5},
    'B': {'D': 2},
    'C': {'G': 3},
    'D': {'G': 6},
    'G': {}
}

# Heurística h(n) definida en la consigna
heuristica = {'S': 7, 'A': 5, 'B': 7, 'C': 3, 'D': 6, 'G': 0}

def voraz(inicio, objetivo):
    frontera = []
    orden_insercion = 0

    # El nodo almacena: estado, padre, g (costo) y h (heurística)
    raiz = {"estado": inicio, "padre": None, "g": 0, "h": heuristica[inicio]}
    mejor_g = {inicio: 0}

    # frontera guarda tuplas: (prioridad h, orden, nodo)
    heapq.heappush(frontera, (raiz["h"], orden_insercion, raiz))
    orden_insercion += 1

    expandidos = 0
    print("--- Traza de Expansión Voraz ---")

    while frontera:
        # Extraemos el nodo con menor prioridad h (y desempate por inserción)
        h_actual, _, nodo = heapq.heappop(frontera)
        estado = nodo["estado"]

        print(f"Extraído: {estado} | h: {nodo['h']} | Costo acumulado hasta acá: {nodo['g']}")
        expandidos += 1

        # El objetivo se comprueba al extraer
        if estado == objetivo:
            camino = []
            actual = nodo
            while actual:
                camino.append(actual["estado"])
                actual = actual["padre"]
            return camino[::-1], nodo["g"], expandidos

        # Relajar transiciones
        for vecino, costo_arista in grafo.get(estado, {}).items():
            nuevo_g = nodo["g"] + costo_arista

            # Según el pseudocódigo, insertamos si mejora el mejor g conocido
            if nuevo_g < mejor_g.get(vecino, float('inf')):
                mejor_g[vecino] = nuevo_g
                nuevo_nodo = {"estado": vecino, "padre": nodo, "g": nuevo_g, "h": heuristica[vecino]}

                # La diferencia clave con UCS: insertamos usando h como prioridad
                heapq.heappush(frontera, (nuevo_nodo["h"], orden_insercion, nuevo_nodo))
                orden_insercion += 1

    return None, 0, expandidos

camino, costo, expandidos = voraz('S', 'G')
print(f"\nResultado -> Camino: {camino} | Costo total: {costo} | Estados expandidos: {expandidos}")

--- Traza de Expansión Voraz ---
Extraído: S | h: 7 | Costo acumulado hasta acá: 0
Extraído: A | h: 5 | Costo acumulado hasta acá: 2
Extraído: C | h: 3 | Costo acumulado hasta acá: 4
Extraído: G | h: 0 | Costo acumulado hasta acá: 7

Resultado -> Camino: ['S', 'A', 'C', 'G'] | Costo total: 7 | Estados expandidos: 4


In [ ]:
import heapq

grafo = {
    'S': {'A': 2, 'B': 2},
    'A': {'C': 2, 'D': 5},
    'B': {'D': 2},
    'C': {'G': 3},
    'D': {'G': 6},
    'G': {}
}

heuristica = {'S': 7, 'A': 5, 'B': 7, 'C': 3, 'D': 6, 'G': 0}

def a_estrella(inicio, objetivo):
    frontera = []
    orden_insercion = 0

    # El nodo almacena: estado, padre, g, h y f
    h_inicial = heuristica[inicio]
    raiz = {"estado": inicio, "padre": None, "g": 0, "h": h_inicial, "f": h_inicial}
    mejor_g = {inicio: 0}

    # frontera guarda tuplas: (prioridad f, orden, nodo)
    heapq.heappush(frontera, (raiz["f"], orden_insercion, raiz))
    orden_insercion += 1

    expandidos = 0
    print("--- Traza de Expansión A* ---")

    while frontera:
        # Extraemos el nodo con menor f = g + h (desempate por inserción)
        f_actual, _, nodo = heapq.heappop(frontera)
        estado = nodo["estado"]

        # Descartar si su g es obsoleto frente a mejor_g
        if nodo["g"] > mejor_g.get(estado, float('inf')):
            continue

        print(f"Extraído: {estado} | f: {nodo['f']} (g:{nodo['g']} + h:{nodo['h']})")
        expandidos += 1

        # Prueba de objetivo al extraer
        if estado == objetivo:
            camino = []
            actual = nodo
            while actual:
                camino.append(actual["estado"])
                actual = actual["padre"]
            return camino[::-1], nodo["g"], expandidos

        # Relajar sucesores e insertar mejoras
        for vecino, costo_arista in grafo.get(estado, {}).items():
            nuevo_g = nodo["g"] + costo_arista

            if nuevo_g < mejor_g.get(vecino, float('inf')):
                mejor_g[vecino] = nuevo_g
                h_vecino = heuristica[vecino]
                nuevo_f = nuevo_g + h_vecino

                nuevo_nodo = {"estado": vecino, "padre": nodo, "g": nuevo_g, "h": h_vecino, "f": nuevo_f}
                heapq.heappush(frontera, (nuevo_f, orden_insercion, nuevo_nodo))
                orden_insercion += 1

    return None, 0, expandidos

camino, costo, expandidos = a_estrella('S', 'G')
print(f"\nResultado -> Camino: {camino} | Costo total: {costo} | Estados expandidos: {expandidos}")

--- Traza de Expansión A* ---
Extraído: S | f: 7 (g:0 + h:7)
Extraído: A | f: 7 (g:2 + h:5)
Extraído: C | f: 7 (g:4 + h:3)
Extraído: G | f: 7 (g:7 + h:0)

Resultado -> Camino: ['S', 'A', 'C', 'G'] | Costo total: 7 | Estados expandidos: 4


In [ ]:
GITHUB_TOKEN = "ghp_tu_token_gigante_aqui"
USUARIO = "ElAgus1"
REPO = "clases_ds_ia"

# Clona el repo usando tu token para tener permisos de escritura
!git clone https://github.com/ElAgus1/clases_ds_ia.git
# Nos movemos adentro de la carpeta del repo
%cd {REPO}